### 1. Εγκατάσταση βιβλιοθηκών

In [ ]:
!pip install transformers datasets torchmetrics

### 2. Imports και ρυθμίσεις

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, AutoModel

from datasets import load_dataset, Audio, DatasetDict

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ORIGINAL_MODEL = "facebook/data2vec-audio-base-960h"                          #προεκπαιδευμένο μοντέλο
STAGES_REPO    = "VasilisAsim/data2vec-finetuned-TESS-v2"                     #M0, M1, M4 (branches)
FINAL_REPO     = "VasilisAsim/data2vec-finetuned-TESS_full"                   #Baseline model
EPOCH_REPO_ID  = "VasilisAsim/data2vec-finetuned-TESS-per-checkpoint-ft-v2"   #M3 ανά epoch (branches)
ENCODER_ATTR   = "data2vec_audio"

### 3. Ορισμός checkpoints

In [ ]:
FINAL_REPO = "VasilisAsim/data2vec-finetuned-TESS_full"      # στο κελί 2, δίπλα στα άλλα repos

CHECKPOINTS = [
    {"name": "M0_frozen",  "source": STAGES_REPO, "revision": "M0-frozen-head"},
    {"name": "M1_partial", "source": STAGES_REPO, "revision": "M1-partial"},
] + [
    {"name": f"M3_epoch{e}", "source": EPOCH_REPO_ID, "revision": f"M3-epoch-{e}"}
    for e in range(1, 9)
] + [
    {"name": "M4_best",         "source": STAGES_REPO, "revision": "M2-full"},   # τελικό του σταδιακού πειράματος
    {"name": "Baseline_final",  "source": FINAL_REPO,  "revision": None},        # μοντέλο αναφοράς (main του _full)
]

STAGE_NAMES = ["M0_frozen", "M1_partial", "M4_best", "Baseline_final"]
EPOCH_NAMES = [f"M3_epoch{e}" for e in range(1, 9)]
STAGE_COLORS = {"original": "tab:gray", "M0_frozen": "tab:blue", "M1_partial": "tab:orange",
                "M4_best": "tab:green", "Baseline_final": "tab:red"}

### 4. Φόρτωση TESS και split ανεξάρτητο ομιλητή

In [ ]:
from datasets import load_dataset, Audio, DatasetDict


def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})


dataset = load_tess_speaker_independent()
train_dataset = dataset["train"]        # OAF
test_dataset = dataset["test"]          # YAF

print(f"Train (OAF): {len(train_dataset)} δείγματα")
print(f"Test  (YAF): {len(test_dataset)} δείγματα")

### 5. Συναρτήσεις φόρτωσης μοντέλου και εξαγωγής embeddings

In [ ]:
def load_for_extraction(source, revision=None):
    """Φορτώνει feature extractor + μοντέλο. Επιστρέφει (model, feature_extractor, encoder_attr).
    - Το προεκπαιδευμένο μοντέλο φορτώνεται ως AutoModel (χωρίς κεφαλή ταξινόμησης).
    - Τα fine-tuned φορτώνονται ως AutoModelForAudioClassification και χρησιμοποιείται ο backbone τους.
    """
    feature_extractor = AutoFeatureExtractor.from_pretrained(source, revision=revision)
    if source == ORIGINAL_MODEL:
        model, encoder_attr = AutoModel.from_pretrained(source), None
    else:
        model, encoder_attr = AutoModelForAudioClassification.from_pretrained(source, revision=revision), ENCODER_ATTR
    return model.to(device).eval(), feature_extractor, encoder_attr


@torch.no_grad()
def extract_embeddings(model, feature_extractor, dataset, encoder_attr=None):
    """
    Mean pooling στον χρόνο των hidden states κάθε layer, ένα δείγμα τη φορά (χωρίς padding).
    Επιστρέφει λίστα [layer] -> np.array (δείγματα x hidden_dim). Layer 0 = έξοδος embeddings.
    """
    backbone = getattr(model, encoder_attr) if encoder_attr else model
    n_layers = model.config.num_hidden_layers + 1
    buffers = [[] for _ in range(n_layers)]

    for sample in dataset:
        audio = sample["audio"]
        inputs = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"],
                                   return_tensors="pt").to(device)
        out = backbone(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(out.hidden_states):
            buffers[layer].append(h.squeeze(0).mean(dim=0).cpu().numpy())

    return [np.stack(b) for b in buffers]

### 6. Γραμμικό CKA

In [ ]:
def linear_CKA(X, Y):
    """
    Γραμμική Centered Kernel Alignment μεταξύ δύο πινάκων αναπαραστάσεων
    (δείγματα × hidden_dim), για το ίδιο layer σε δύο διαφορετικά checkpoints.
    """
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)

    hsic = np.linalg.norm(X.T @ Y, ord="fro") ** 2
    var1 = np.linalg.norm(X.T @ X, ord="fro")
    var2 = np.linalg.norm(Y.T @ Y, ord="fro")

    return hsic / (var1 * var2)

### 7. Εξαγωγή embeddings και υπολογισμός CKA για τα δύο μοντέλα

In [ ]:
# Embeddings του προεκπαιδευμένου μοντέλου (σταθερό σημείο αναφοράς, υπολογίζεται μία φορά)
print("Εξαγωγή embeddings: original ...")
model, fe, attr = load_for_extraction(ORIGINAL_MODEL)
emb_pre = extract_embeddings(model, fe, test_dataset, attr)
del model
torch.cuda.empty_cache()

# CKA ανά layer για κάθε checkpoint. Τα embeddings του checkpoint δεν κρατιούνται, για οικονομία μνήμης.
cka_rows = []
for ckpt in CHECKPOINTS:
    print(f"Εξαγωγή embeddings: {ckpt['name']} ({ckpt['source']}@{ckpt['revision']}) ...")
    model, fe, attr = load_for_extraction(ckpt["source"], ckpt["revision"])
    emb_ckpt = extract_embeddings(model, fe, test_dataset, attr)
    del model
    torch.cuda.empty_cache()

    for layer in range(len(emb_pre)):
        cka_rows.append({"checkpoint": ckpt["name"], "layer": layer,
                         "cka": linear_CKA(emb_pre[layer], emb_ckpt[layer])})

cka_results = pd.DataFrame(cka_rows)
cka_results.to_csv("cka_results_data2vec_tess.csv", index=False)

### 8. Πίνακας αποτελεσμάτων (layers × checkpoints)

In [ ]:
cka_table = cka_results.pivot(index="layer", columns="checkpoint", values="cka")
cka_table = cka_table[[c["name"] for c in CHECKPOINTS]]      # σειρά όπως στη λίστα checkpoints
cka_table.round(3)

### 9. Οπτικοποίηση 1: CKA ανά στάδιο (M0 / M1 / M4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name in STAGE_NAMES:
    sub = cka_results[cka_results["checkpoint"] == name]
    ax.plot(sub["layer"], sub["cka"], marker="o", label=name, color=STAGE_COLORS[name])

ax.set_xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
ax.set_ylabel("CKA similarity (Pretrained vs Checkpoint)")
ax.set_title("Layer-wise CKA ανά στάδιο (data2vec, TESS)")
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cka_stages_data2vec_tess.png", dpi=300)
plt.show()

### 10. Οπτικοποίηση 2: CKA ανά epoch (στάδιο M3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(EPOCH_NAMES)))
for name, color in zip(EPOCH_NAMES, colors):
    sub = cka_results[cka_results["checkpoint"] == name]
    ax.plot(sub["layer"], sub["cka"], marker="o", label=name, color=color)

ax.set_xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
ax.set_ylabel("CKA similarity (Pretrained vs Checkpoint)")
ax.set_title("Layer-wise CKA ανά epoch (data2vec, TESS, M3)")
ax.set_ylim(0, 1.02)
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cka_epochs_data2vec_tess.png", dpi=300)
plt.show()